# KataGo Remote Engine on Modal

This notebook deploys a KataGo WebSocket server on Modal so KaTrain/SWHub can connect to it with a `wss://.../katago` URL.

## What To Pick

If you are new, use these defaults first:

| Setting | Beginner Choice | What it means |
| --- | --- | --- |
| `ENGINE_MODE` | `"gpu"` | Uses a Modal GPU. This is the useful/fast option. |
| `MODAL_GPU` | `"T4"` | Cheapest useful GPU; stretches the free monthly credit. |
| `MAIN_MODEL_PRESET` | `"transformer_medium_v1_17"` | Recommended v1.17 transformer default: a stronger balance for remote GPU analysis. |
| `HUMAN_MODEL_PRESET` | `"official_human_sl_v0"` | Enables KaTrain/SWHub human/rank-style and career features by default. |

Use CPU mode only for testing:

```python
ENGINE_MODE = "cpu"
```

CPU mode downloads the KataGo Eigen AVX2 build and does **not** request a Modal GPU, but it will be much slower. Use low visits in KaTrain/SWHub.

## Basic Steps

1. Run **Install Modal**.
2. Run **Login** and follow the browser/device-code instructions.
3. Run **Write the Modal App**. This creates `modal_katago_server.py` in the notebook working directory.
4. Run **Deploy**.
5. Copy the printed `https://...modal.run` URL.
6. Convert it to `wss://...modal.run/katago` and paste that into KaTrain/SWHub.
7. Open `https://...modal.run/monitor` to watch logs.

## Cost Safety

Modal Starter includes monthly free compute credits, but GPU time is still metered. When finished, stop the app using the final cell or the Modal dashboard.


## 1. Install Modal

Run this once in the notebook environment. If it says Modal is already installed, that is fine.


In [ ]:
%uv pip install -q modal==1.5.4

## 2. Login

Run this and follow Modal's login flow. It may print a link or device code.


In [ ]:
import os, re, subprocess

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"

p = subprocess.Popen(
    ["modal", "token", "new"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)

for line in p.stdout:
    print(line, end="")
    m = re.search(r"https://modal\.com/token-flow/[A-Za-z0-9_-]+", line)
    if m:
        print("\n\nOPEN THIS LINK NOW:")
        print(m.group(0))


## 3. Write the Modal App

This writes `modal_katago_server.py` with the settings and server code.

To change GPU/CPU or the main model, edit the settings near the top of the generated file after running this cell, or edit the same settings inside this cell before running it.

Most useful edits:

```python
ENGINE_MODE = "gpu"          # or "cpu"
MODAL_GPU = "T4"             # T4, L4, or A10
MAIN_MODEL_PRESET = "transformer_medium_v1_17"  # recommended
HUMAN_MODEL_PRESET = "official_human_sl_v0"
```


In [ ]:
%%writefile modal_katago_server.py
import asyncio
import json
import hashlib
import os
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from collections import deque
from pathlib import Path
from typing import Dict, Tuple

import modal

# =========================
# User settings
# =========================

# Pick "gpu" for KataGo CUDA on a Modal GPU, or "cpu" for the cheaper CPU AVX2 build.
# Start with "gpu" if you want useful speed. Use "cpu" only for testing or very low visits.
ENGINE_MODE = "gpu"

# Cheapest useful GPU choices to try: "T4", "L4", "A10".
# T4 stretches free credits the furthest, but may be slower. Ignored in CPU mode.
MODAL_GPU = "T4"

# Main model preset: "transformer_small_v1_17", "transformer_medium_v1_17", "transformer_large_v1_17", an older convolutional preset, or "custom".
MAIN_MODEL_PRESET = "transformer_medium_v1_17"
MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_FILENAME = "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz"
MODEL_SHA256 = "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995"

# Human SL is optional and costs extra startup/download/storage time.
# Use "official_human_sl_v0" if KaTrain/SWHub features need humanSLProfile/rank-style analysis.
HUMAN_MODEL_PRESET = "official_human_sl_v0"
ENABLE_HUMAN_MODEL = True
HUMAN_MODEL_URL = "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_FILENAME = "b18c384nbt-humanv0.bin.gz"
HUMAN_MODEL_SHA256 = "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5"

KATAGO_RELEASE_CACHE_KEY = "v1_17_1"
GPU_CUDA_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.1/katago-v1.17.1-cuda12.8-cudnn9.8.0-linux-x64.zip"
GPU_CUDA_KATAGO_SHA256 = "458d226c2c8533600251bba3b2ee612d3aee0c796f592a2b53839a6a05b0826e"
CPU_AVX2_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.1/katago-v1.17.1-eigenavx2-linux-x64.zip"
CPU_AVX2_KATAGO_SHA256 = "234bf7866bc26f37baaeed60dc358b821bafc8e73e9bc50cb2d2a1cf51502d44"
CPU_PLAIN_KATAGO_URL = "https://github.com/lightvector/KataGo/releases/download/v1.17.1/katago-v1.17.1-eigen-linux-x64.zip"
CPU_PLAIN_KATAGO_SHA256 = "cca71fff39abd19bd9acfc17750025d4bb0ee6adbad99d7513a2c6401b0a7af3"
SERVER_PORT = 8000
KATAGO_WS_PATH = "/katago"

GPU_MAX_VISITS = 500
GPU_SEARCH_THREADS = 4
GPU_ANALYSIS_THREADS = 4
GPU_NN_CACHE_POWER = 21
GPU_NN_MAX_BATCH_SIZE = 16
GPU_CUDA_DEVICE = 0

CPU_MAX_VISITS = 50
CPU_SEARCH_THREADS = 2
CPU_ANALYSIS_THREADS = 1
CPU_NN_CACHE_POWER = 18
CPU_NN_MAX_BATCH_SIZE = 1

SUBPROCESS_BUFFER_MB = 50
MONITOR_LOG_LIMIT = 1000

# =========================
# Presets
# =========================

MAIN_MODEL_PRESETS = {
    "transformer_small_v1_17": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c384h6nbttflrs.bin.gz",
        "filename": "b10c384h6nbttflrs.bin.gz",
        "sha256": "0ba27eced5180b3e3d0b898b280c541112989765e789d1eb6cd0d31b2b2c1229",
    },
    "transformer_medium_v1_17": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "filename": "b10c512h8nbt3tflrs-fson-silu-rsnh.bin.gz",
        "sha256": "c04db4a503721d948bb720324f3cbdac6088cc9eb243632f020e4b6846f58995",
    },
    "transformer_large_v1_17": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.17.0/b11c768h12nbt3tflrs-fson-silu.bin.gz",
        "filename": "b11c768h12nbt3tflrs-fson-silu.bin.gz",
        "sha256": "1881600caab9e9d85a3dd6a019e9b8e7d2c237b5f984e13ed49a8645be3077c6",
    },
    "modal_fast_b18": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "filename": "kata1-b18c384nbt-s9996604416-d4316597426.bin.gz",
        "sha256": "9d7a6afed8ff5b74894727e156f04f0cd36060a24824892008fbb6e0cba51f1d",
    },
    "latest_b28": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "filename": "kata1-b28c512nbt-s13255194368-d5935380940.bin.gz",
        "sha256": "c5bca453d7b08ea8df6546439325d4dd681e77d975e1da3f7593d771147b73bc",
    },
    "strongest_b40_zhizi": {
        "url": "https://media.katagotraining.org/uploaded/networks/models/kata1/kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "filename": "kata1-zhizi-b40c768nbt-s11272M-d5935M.bin.gz",
        "sha256": "15fb3baf85cdb6578e6c19b65e6201ca906cec3ba5fee19039d05221d57eb0e8",
    },
}

HUMAN_MODEL_PRESETS = {
    "official_human_sl_v0": {
        "url": "https://github.com/lightvector/KataGo/releases/download/v1.15.0/b18c384nbt-humanv0.bin.gz",
        "filename": "b18c384nbt-humanv0.bin.gz",
        "sha256": "637746e44f0efe00ad1245a50aa9bbf0716efe364c43965ead97bd6835d84ab5",
    },
}


def apply_model_presets():
    global MODEL_URL, MODEL_FILENAME, MODEL_SHA256
    global ENABLE_HUMAN_MODEL, HUMAN_MODEL_URL, HUMAN_MODEL_FILENAME, HUMAN_MODEL_SHA256

    main = MAIN_MODEL_PRESET.lower().strip()
    if main != "custom":
        if main not in MAIN_MODEL_PRESETS:
            raise ValueError(f"MAIN_MODEL_PRESET must be one of {sorted([*MAIN_MODEL_PRESETS, 'custom'])}")
        MODEL_URL = MAIN_MODEL_PRESETS[main]["url"]
        MODEL_FILENAME = MAIN_MODEL_PRESETS[main]["filename"]
        MODEL_SHA256 = MAIN_MODEL_PRESETS[main]["sha256"]

    human = HUMAN_MODEL_PRESET.lower().strip()
    if human == "disabled":
        ENABLE_HUMAN_MODEL = False
    elif human != "custom":
        if human not in HUMAN_MODEL_PRESETS:
            raise ValueError(f"HUMAN_MODEL_PRESET must be one of {sorted([*HUMAN_MODEL_PRESETS, 'custom', 'disabled'])}")
        ENABLE_HUMAN_MODEL = True
        HUMAN_MODEL_URL = HUMAN_MODEL_PRESETS[human]["url"]
        HUMAN_MODEL_FILENAME = HUMAN_MODEL_PRESETS[human]["filename"]
        HUMAN_MODEL_SHA256 = HUMAN_MODEL_PRESETS[human]["sha256"]
    else:
        ENABLE_HUMAN_MODEL = True


apply_model_presets()

ENGINE_MODE = ENGINE_MODE.lower().strip()
if ENGINE_MODE not in {"gpu", "cpu"}:
    raise ValueError('ENGINE_MODE must be "gpu" or "cpu"')

USE_GPU = ENGINE_MODE == "gpu"
KATAGO_URL = GPU_CUDA_KATAGO_URL if USE_GPU else CPU_AVX2_KATAGO_URL
KATAGO_SHA256 = GPU_CUDA_KATAGO_SHA256 if USE_GPU else CPU_AVX2_KATAGO_SHA256
MODE_LABEL = f"Modal GPU {MODAL_GPU} CUDA" if USE_GPU else "Modal CPU AVX2"

APP_NAME = "KataGo-Analyse"
app = modal.App(APP_NAME)
volume = modal.Volume.from_name("katago", create_if_missing=True)

base_image = (
    modal.Image.from_registry("nvidia/cuda:12.8.1-cudnn-runtime-ubuntu22.04", add_python="3.11").entrypoint([])
    if USE_GPU
    else modal.Image.debian_slim(python_version="3.11")
)

image = (
    base_image
    .apt_install(
        "wget",
        "unzip",
        "p7zip-full",
        "libzip4",
        "libomp-dev",
        "libgomp1",
        "ca-certificates",
        "curl",
    )
    .pip_install("fastapi[standard]==0.141.1", "uvicorn==0.52.4", "websockets==17.0.1")
)

CACHE = Path("/cache")
KATAGO_DIR = CACHE / (f"katago_{KATAGO_RELEASE_CACHE_KEY}_gpu_cuda12_8_cudnn9" if USE_GPU else f"katago_{KATAGO_RELEASE_CACHE_KEY}_cpu")
MODEL_DIR = KATAGO_DIR / "models"
KATAGO_BIN = KATAGO_DIR / "katago"
KATAGO_EXTRACTED_BIN = KATAGO_DIR / "squashfs-root" / "AppRun"
CONFIG_PATH = KATAGO_DIR / "analysis.cfg"
MODEL_PATH = MODEL_DIR / MODEL_FILENAME
HUMAN_MODEL_PATH = MODEL_DIR / HUMAN_MODEL_FILENAME


def run(cmd, check=True, cwd=None):
    print(f"\n$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    print(result.stdout, flush=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stdout}")
    return result.stdout


def command_output(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd)
    return result.returncode, result.stdout


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_download(path, expected_sha256):
    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Download produced an empty file: {path}")
    if expected_sha256:
        actual = sha256_file(path)
        if actual.lower() != expected_sha256.lower():
            raise RuntimeError(f"SHA-256 mismatch for {path.name}: expected {expected_sha256}, got {actual}")


def download(url, path, expected_sha256=""):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        try:
            verify_download(path, expected_sha256)
            print(f"Already exists and verified: {path}", flush=True)
            return
        except Exception as exc:
            print(f"Cached file failed verification; downloading again: {exc}", flush=True)
            path.unlink(missing_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    print(f"Downloading {url} -> {path}", flush=True)

    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 KataGo-Modal/1.0",
        "Accept": "*/*",
    }
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=120) as response, open(tmp, "wb") as out:
            shutil.copyfileobj(response, out)
    except Exception as exc:
        print(f"Python download failed: {exc!r}", flush=True)
        print("Trying wget fallback...", flush=True)
        tmp.unlink(missing_ok=True)
        run(
            f'wget --tries=3 --timeout=60 --user-agent="Mozilla/5.0 KataGo-Modal/1.0" -O "{tmp}" "{url}"',
            check=True,
        )

    try:
        verify_download(tmp, expected_sha256)
        os.replace(tmp, path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


def ensure_katago_files():
    KATAGO_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    if not KATAGO_BIN.exists():
        zip_path = KATAGO_DIR / "katago.zip"
        download(KATAGO_URL, zip_path, KATAGO_SHA256)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(KATAGO_DIR)
        KATAGO_BIN.chmod(0o755)
        zip_path.unlink(missing_ok=True)

    download(MODEL_URL, MODEL_PATH, MODEL_SHA256)
    if ENABLE_HUMAN_MODEL:
        download(HUMAN_MODEL_URL, HUMAN_MODEL_PATH, HUMAN_MODEL_SHA256)

    if USE_GPU:
        config = f"""logDir = {KATAGO_DIR}/logs

maxVisits = {GPU_MAX_VISITS}

numSearchThreadsPerAnalysisThread = {GPU_SEARCH_THREADS}
numAnalysisThreads = {GPU_ANALYSIS_THREADS}

nnCacheSizePowerOfTwo = {GPU_NN_CACHE_POWER}
nnMaxBatchSize = {GPU_NN_MAX_BATCH_SIZE}
numNNServerThreadsPerModel = 1

cudaDeviceToUseThread0 = {GPU_CUDA_DEVICE}
"""
    else:
        config = f"""logDir = {KATAGO_DIR}/logs

maxVisits = {CPU_MAX_VISITS}

numSearchThreadsPerAnalysisThread = {CPU_SEARCH_THREADS}
numAnalysisThreads = {CPU_ANALYSIS_THREADS}

nnCacheSizePowerOfTwo = {CPU_NN_CACHE_POWER}
nnMaxBatchSize = {CPU_NN_MAX_BATCH_SIZE}
numNNServerThreadsPerModel = 1
"""

    CONFIG_PATH.write_text(config)

    print(f'\n$ "{KATAGO_BIN}" version', flush=True)
    code, out = command_output(f'"{KATAGO_BIN}" version')
    print(out, flush=True)
    if code == 0 and re.search(r"(?:^|[^0-9])1\.17\.1(?:[^0-9]|$)", out):
        katago_exec = KATAGO_BIN
    else:
        if code == 0:
            raise RuntimeError(f"Expected KataGo 1.17.1, got:\n{out}")
        print("KataGo did not run directly. Trying AppImage extraction fallback...", flush=True)
        if not KATAGO_EXTRACTED_BIN.exists():
            run(f'"{KATAGO_BIN}" --appimage-extract', check=True, cwd=KATAGO_DIR)
        if not KATAGO_EXTRACTED_BIN.exists():
            raise RuntimeError(f"AppImage extraction did not create {KATAGO_EXTRACTED_BIN}")
        KATAGO_EXTRACTED_BIN.chmod(0o755)
        fallback_out = run(f'"{KATAGO_EXTRACTED_BIN}" version', check=True)
        if not re.search(r"(?:^|[^0-9])1\.17\.1(?:[^0-9]|$)", fallback_out):
            raise RuntimeError(f"Expected KataGo 1.17.1, got:\n{fallback_out}")
        katago_exec = KATAGO_EXTRACTED_BIN

    if USE_GPU:
        run("nvidia-smi", check=False)
    return katago_exec


function_options = {
    "image": image,
    "volumes": {"/cache": volume},
    "timeout": 60 * 60 * 4,
    "cpu": 4.0 if not USE_GPU else 2.0,
    "memory": 8192 if not USE_GPU else 4096,
    "max_containers": 1,
    "scaledown_window": 60,
}
if USE_GPU:
    function_options["gpu"] = MODAL_GPU


@app.function(**function_options)
@modal.concurrent(max_inputs=20)
@modal.asgi_app()
def go():
    from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request
    from fastapi.responses import HTMLResponse, PlainTextResponse, StreamingResponse

    web_app = FastAPI()
    katago_proc = None
    pending: Dict[str, Tuple[WebSocket, asyncio.Lock]] = {}
    pending_expected_turns: Dict[str, set] = {}
    pending_completed_turns: Dict[str, set] = {}
    katago_write_lock = asyncio.Lock()
    logs = deque(maxlen=MONITOR_LOG_LIMIT)
    event_subscribers = set()
    stale_result_ids = set()
    stale_result_order = deque()

    state = {
        "mode": MODE_LABEL,
        "started_at": time.time(),
        "katago_started": False,
        "katago_ready": False,
        "websocket_clients_total": 0,
        "websocket_clients_active": 0,
        "queries_received": 0,
        "results_sent": 0,
        "errors": 0,
        "warnings": 0,
        "last_error": None,
        "last_query_id": None,
        "last_result_id": None,
        "last_best_move": None,
        "last_visits": None,
        "last_winrate": None,
        "last_score_lead": None,
        "pending_queries": 0,
        "stale_results_dropped": 0,
        "human_model_enabled": ENABLE_HUMAN_MODEL,
    }

    def now_time():
        return time.strftime("%H:%M:%S")

    async def emit(event_type, message, extra=None):
        if event_type == "error":
            state["errors"] += 1
            state["last_error"] = str(message)
        if event_type == "warning":
            state["warnings"] += 1
        state["pending_queries"] = len(pending)
        item = {"ts": time.time(), "time": now_time(), "type": event_type, "message": str(message), "state": dict(state)}
        if extra:
            item.update(extra)
        logs.append(item)
        dead = []
        for q in event_subscribers:
            try:
                q.put_nowait(item)
            except Exception:
                dead.append(q)
        for q in dead:
            event_subscribers.discard(q)
        print(f"[{item['time']}] {event_type}: {message}", flush=True)

    def first_stale_result(query_id):
        if not query_id or query_id in stale_result_ids:
            return False
        stale_result_ids.add(query_id)
        stale_result_order.append(query_id)
        while len(stale_result_order) > 500:
            stale_result_ids.discard(stale_result_order.popleft())
        return True

    def finish_pending_query(query_id):
        pending.pop(query_id, None)
        pending_expected_turns.pop(query_id, None)
        pending_completed_turns.pop(query_id, None)

    async def terminate_pending_query(query_id, reason):
        if query_id not in pending:
            return
        finish_pending_query(query_id)
        try:
            await write_to_katago({"id": f"terminate-{query_id}-{time.time_ns()}", "action": "terminate", "terminateId": query_id})
            await emit("query", f"Terminated previous pending query id={query_id}: {reason}", {"query_id": query_id})
        except Exception as exc:
            await emit("error", f"Could not terminate query id={query_id}: {exc!r}")

    async def write_to_katago(payload):
        if katago_proc is None or katago_proc.stdin is None:
            raise RuntimeError("KataGo process is not running")
        if katago_proc.returncode is not None:
            raise RuntimeError(f"KataGo process exited with code {katago_proc.returncode}")
        async with katago_write_lock:
            katago_proc.stdin.write((json.dumps(payload) + "\n").encode("utf-8"))
            await katago_proc.stdin.drain()

    async def start_katago():
        nonlocal katago_proc
        try:
            await emit("system", "Preparing KataGo files in Modal volume...")
            katago_exec = ensure_katago_files()
            try:
                volume.commit()
            except Exception as exc:
                await emit("warning", f"Volume commit warning: {exc!r}")

            args = [str(katago_exec), "analysis", "-model", str(MODEL_PATH), "-config", str(CONFIG_PATH)]
            if ENABLE_HUMAN_MODEL:
                args.extend(["-human-model", str(HUMAN_MODEL_PATH)])
            else:
                await emit("warning", "Human SL model disabled; humanSLProfile queries will fail.")

            await emit("system", "Launch command: " + " ".join(args))
            katago_proc = await asyncio.create_subprocess_exec(
                *args,
                stdin=asyncio.subprocess.PIPE,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                limit=SUBPROCESS_BUFFER_MB * 1024 * 1024,
            )
            state["katago_started"] = True
            asyncio.create_task(read_stdout())
            asyncio.create_task(read_stderr())
            await emit("system", "KataGo process started")
        except Exception as exc:
            await emit("error", f"KataGo startup failed: {exc!r}")

    async def read_stdout():
        while True:
            line = await katago_proc.stdout.readline()
            if not line:
                await emit("system", "KataGo stdout closed")
                break
            text = line.decode("utf-8", errors="ignore").strip()
            if not text:
                continue
            try:
                data = json.loads(text)
                query_id = data.get("id")
                terminate_id = data.get("terminateId")
                if terminate_id:
                    await emit("result", f"Termination acknowledged id={terminate_id}")
                    continue
                if query_id is None and (data.get("error") or data.get("warning") or data.get("noResults")):
                    detail = data.get("error") or data.get("warning") or "KataGo returned no results"
                    level = "error" if data.get("error") or data.get("noResults") else "warning"
                    await emit(level, f"Unscoped KataGo {level}: {detail}")
                    delivered = set()
                    for websocket, send_lock in list(pending.values()):
                        if id(websocket) in delivered:
                            continue
                        delivered.add(id(websocket))
                        try:
                            async with send_lock:
                                await websocket.send_text(json.dumps(data))
                        except Exception as exc:
                            await emit("warning", f"Could not forward unscoped KataGo error: {exc!r}")
                    continue
                target = pending.get(query_id)
                if not target:
                    state["stale_results_dropped"] += 1
                    if first_stale_result(query_id):
                        await emit("stale", f"Dropped stale result id={query_id}; client already cancelled or replaced this query")
                    continue
                websocket, send_lock = target
                async with send_lock:
                    await websocket.send_text(json.dumps(data))
                state["results_sent"] += 1
                state["last_result_id"] = query_id
                move_infos = data.get("moveInfos", [])
                if move_infos:
                    state["last_best_move"] = move_infos[0].get("move")
                    state["last_visits"] = move_infos[0].get("visits")
                    state["last_winrate"] = move_infos[0].get("winrate")
                    state["last_score_lead"] = move_infos[0].get("scoreLead")
                await emit("result", f"Sent result id={query_id}, best={state['last_best_move']}, visits={state['last_visits']}")
                if data.get("error") or data.get("noResults"):
                    finish_pending_query(query_id)
                elif data.get("isDuringSearch") is False:
                    expected = pending_expected_turns.get(query_id, set())
                    if not expected:
                        finish_pending_query(query_id)
                    else:
                        turn_number = data.get("turnNumber")
                        if turn_number is not None:
                            pending_completed_turns.setdefault(query_id, set()).add(int(turn_number))
                        if expected.issubset(pending_completed_turns.get(query_id, set())):
                            finish_pending_query(query_id)
            except Exception as exc:
                await emit("error", f"Error forwarding KataGo output: {exc!r}")

    async def read_stderr():
        while True:
            line = await katago_proc.stderr.readline()
            if not line:
                break
            msg = line.decode("utf-8", errors="ignore").rstrip()
            lower = msg.lower()
            if "started, ready to begin handling requests" in lower:
                state["katago_ready"] = True
                await emit("katago", "KataGo ready")
            elif "warning" in lower:
                await emit("warning", msg)
            elif "error" in lower or "failed" in lower:
                await emit("error", msg)
            elif "tuning" in lower or "calls/sec" in lower or "done tuning" in lower:
                await emit("tuning", msg)
            else:
                print("KataGo:", msg, flush=True)

    @web_app.on_event("startup")
    async def startup():
        asyncio.create_task(start_katago())
        await emit("system", "Monitor is online. KataGo is starting in the background.")

    @web_app.on_event("shutdown")
    async def shutdown():
        if katago_proc and katago_proc.returncode is None:
            katago_proc.terminate()

    @web_app.get("/")
    async def home():
        return {"ok": True, "message": "KataGo Modal server running", "websocket": KATAGO_WS_PATH, "monitor": "/monitor"}

    @web_app.get("/status")
    async def status():
        state["pending_queries"] = len(pending)
        return {"ok": True, "state": state, "recent_logs": list(logs)[-50:]}

    @web_app.get("/logs", response_class=PlainTextResponse)
    async def get_logs(type: str = "all", q: str = ""):
        rows = list(logs)
        if type and type != "all":
            wanted = {t.strip() for t in type.split(",") if t.strip()}
            rows = [item for item in rows if item.get("type") in wanted]
        if q:
            q_lower = q.lower()
            rows = [item for item in rows if q_lower in item.get("message", "").lower()]
        return "\n".join(f"[{item['time']}] {item['type']}: {item['message']}" for item in rows)

    @web_app.get("/events")
    async def events(request: Request):
        queue = asyncio.Queue()
        event_subscribers.add(queue)

        async def stream():
            try:
                for item in list(logs)[-50:]:
                    yield f"data: {json.dumps(item)}\n\n"
                while True:
                    if await request.is_disconnected():
                        break
                    try:
                        item = await asyncio.wait_for(queue.get(), timeout=15)
                        yield f"data: {json.dumps(item)}\n\n"
                    except asyncio.TimeoutError:
                        yield f"data: {json.dumps({'time': now_time(), 'type': 'heartbeat', 'message': 'alive', 'state': dict(state)})}\n\n"
            finally:
                event_subscribers.discard(queue)

        return StreamingResponse(stream(), media_type="text/event-stream")

    @web_app.get("/monitor", response_class=HTMLResponse)
    async def monitor():
        return HTMLResponse("""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>KataGo Modal Monitor</title>
  <style>
    :root { color-scheme: dark; }
    * { box-sizing: border-box; }
    body { font-family: system-ui, sans-serif; background: #0f1115; color: #e6e6e6; margin: 0; padding: 18px; }
    h1 { margin: 0 0 8px 0; font-size: 24px; }
    h2 { margin: 18px 0 10px; font-size: 18px; }
    .small { color: #9ca3af; font-size: 13px; margin-bottom: 14px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 12px; margin-bottom: 18px; }
    .card { background: #171a21; border: 1px solid #2a2f3a; border-radius: 8px; padding: 12px; min-height: 70px; }
    .label { color: #9ca3af; font-size: 12px; margin-bottom: 6px; }
    .value { font-size: 20px; font-weight: 700; word-break: break-word; }
    .mono { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; font-size: 13px; }
    .ok { color: #61d394; } .bad { color: #ff6b6b; } .warn { color: #ffd166; } .muted { color: #9ca3af; }
    .toolbar { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; background: #171a21; border: 1px solid #2a2f3a; border-radius: 8px; padding: 10px; margin-bottom: 10px; }
    button, input { background: #0f1115; color: #e6e6e6; border: 1px solid #374151; border-radius: 8px; padding: 7px 9px; font: inherit; }
    button { cursor: pointer; }
    button.active { background: #2563eb; border-color: #60a5fa; }
    input { min-width: min(260px, 100%); }
    #log { background: #05070a; border: 1px solid #2a2f3a; border-radius: 8px; padding: 12px; height: 55vh; overflow-y: auto; white-space: pre-wrap; font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; font-size: 13px; line-height: 1.45; }
    .event-system { color: #93c5fd; }
    .event-katago { color: #a7f3d0; }
    .event-tuning { color: #67e8f9; }
    .event-client { color: #fcd34d; }
    .event-query { color: #c4b5fd; }
    .event-result { color: #86efac; }
    .event-stale { color: #9ca3af; }
    .event-error { color: #fca5a5; }
    .event-warning { color: #fdba74; }
    .event-heartbeat { color: #6b7280; }
    code { background: #1f2937; padding: 2px 5px; border-radius: 4px; }
    a { color: #93c5fd; }
  </style>
</head>
<body>
  <h1>KataGo Modal Monitor</h1>
  <div class="small">WebSocket path: <code>/katago</code> | Live endpoint: <code>/events</code> | Logs endpoint: <code>/logs?type=error,warning&q=text</code></div>

  <div class="grid">
    <div class="card"><div class="label">Mode</div><div id="mode" class="value">-</div></div>
    <div class="card"><div class="label">KataGo Ready</div><div id="katago_ready" class="value warn">checking...</div></div>
    <div class="card"><div class="label">Human Model</div><div id="human_model" class="value">-</div></div>
    <div class="card"><div class="label">Active Clients</div><div id="clients" class="value">0</div></div>
    <div class="card"><div class="label">Queries</div><div id="queries" class="value">0</div></div>
    <div class="card"><div class="label">Results</div><div id="results" class="value">0</div></div>
    <div class="card"><div class="label">Pending</div><div id="pending" class="value">0</div></div>
    <div class="card"><div class="label">Dropped</div><div id="dropped" class="value muted">0</div></div>
    <div class="card"><div class="label">Errors / Warnings</div><div class="value"><span id="errors" class="bad">0</span> / <span id="warnings" class="warn">0</span></div></div>
    <div class="card"><div class="label">Last Best Move</div><div id="best_move" class="value">-</div></div>
    <div class="card"><div class="label">Last Visits</div><div id="last_visits" class="value">-</div></div>
    <div class="card"><div class="label">Last Winrate</div><div id="last_winrate" class="value">-</div></div>
    <div class="card"><div class="label">Last Score Lead</div><div id="last_score" class="value">-</div></div>
    <div class="card"><div class="label">Last Query ID</div><div id="last_query" class="value mono">-</div></div>
    <div class="card"><div class="label">Last Result ID</div><div id="last_result" class="value mono">-</div></div>
    <div class="card"><div class="label">Last Error</div><div id="last_error" class="value bad mono">none</div></div>
  </div>

  <h2>Live Events</h2>
  <div class="toolbar">
    <button data-filter="all" class="active">All</button>
    <button data-filter="system">System</button>
    <button data-filter="katago">KataGo</button>
    <button data-filter="tuning">Tuning</button>
    <button data-filter="client">Client</button>
    <button data-filter="query">Query</button>
    <button data-filter="result">Result</button>
    <button data-filter="stale">Dropped</button>
    <button data-filter="warning">Warning</button>
    <button data-filter="error">Error</button>
    <button data-filter="heartbeat">Heartbeat</button>
    <input id="search" placeholder="Search logs...">
    <button id="pause">Pause</button>
    <button id="clear">Clear View</button>
  </div>
  <div id="log"></div>

  <script>
    const logEl = document.getElementById("log");
    const allEvents = [];
    const seenEvents = new Set();
    let currentFilter = "all";
    let paused = false;
    let sseFailures = 0;
    let pollingLogs = false;

    function setText(id, value) { document.getElementById(id).textContent = value ?? "-"; }
    function fmtFloat(x, digits=3) { return typeof x === "number" ? x.toFixed(digits) : "-"; }
    function fmtPct(x) { return typeof x === "number" ? (x * 100).toFixed(1) + "%" : "-"; }

    function updateState(state) {
      if (!state) return;
      setText("mode", state.mode || "-");
      setText("human_model", state.human_model_enabled ? "ON" : "OFF");
      const ready = document.getElementById("katago_ready");
      if (state.katago_ready) { ready.textContent = "YES"; ready.className = "value ok"; }
      else { ready.textContent = "NO"; ready.className = "value warn"; }
      setText("clients", state.websocket_clients_active);
      setText("queries", state.queries_received);
      setText("results", state.results_sent);
      setText("pending", state.pending_queries);
      setText("dropped", state.stale_results_dropped);
      setText("errors", state.errors);
      setText("warnings", state.warnings);
      setText("best_move", state.last_best_move || "-");
      setText("last_visits", state.last_visits ?? "-");
      setText("last_winrate", fmtPct(state.last_winrate));
      setText("last_score", fmtFloat(state.last_score_lead));
      setText("last_query", state.last_query_id || "-");
      setText("last_result", state.last_result_id || "-");
      setText("last_error", state.last_error || "none");
    }

    function passesFilter(item) {
      const q = document.getElementById("search").value.toLowerCase().trim();
      const typeOk = currentFilter === "all" || item.type === currentFilter;
      const text = `${item.type || ""} ${item.message || ""}`.toLowerCase();
      return typeOk && (!q || text.includes(q));
    }

    function renderLogs() {
      logEl.innerHTML = "";
      const visible = allEvents.filter(passesFilter).slice(-500);
      for (const item of visible) {
        const line = document.createElement("div");
        const type = item.type || "event";
        line.className = "event-" + type;
        line.textContent = `[${item.time}] ${type}: ${item.message}`;
        logEl.appendChild(line);
      }
      logEl.scrollTop = logEl.scrollHeight;
    }

    function addEvent(item) {
      updateState(item.state);
      const key = `${item.ts || ""}|${item.time || ""}|${item.type || ""}|${item.message || ""}`;
      if (seenEvents.has(key)) return;
      seenEvents.add(key);
      if (seenEvents.size > 2500) seenEvents.clear();
      allEvents.push(item);
      if (allEvents.length > 2000) allEvents.shift();
      if (!paused) renderLogs();
    }

    async function pollStatus() {
      try {
        const res = await fetch("/status");
        const data = await res.json();
        updateState(data.state);
        if (pollingLogs && Array.isArray(data.recent_logs)) {
          data.recent_logs.forEach(addEvent);
        }
      } catch (e) { console.log(e); }
    }

    document.querySelectorAll("button[data-filter]").forEach(btn => {
      btn.onclick = () => {
        document.querySelectorAll("button[data-filter]").forEach(b => b.classList.remove("active"));
        btn.classList.add("active");
        currentFilter = btn.dataset.filter;
        renderLogs();
      };
    });

    document.getElementById("search").oninput = renderLogs;
    document.getElementById("pause").onclick = () => {
      paused = !paused;
      document.getElementById("pause").textContent = paused ? "Resume" : "Pause";
      if (!paused) renderLogs();
    };
    document.getElementById("clear").onclick = () => { allEvents.length = 0; renderLogs(); };

    const source = new EventSource("/events");
    source.onmessage = function(event) {
      try { addEvent(JSON.parse(event.data)); }
      catch (e) { console.log(e, event.data); }
    };
    source.onerror = function() {
      sseFailures += 1;
      addEvent({time: new Date().toLocaleTimeString(), type: "error", message: "Browser EventSource disconnected/reconnecting...", state: null});
      if (sseFailures >= 3 && !pollingLogs) {
        pollingLogs = true;
        source.close();
        addEvent({time: new Date().toLocaleTimeString(), type: "warning", message: "Live events unavailable; monitor switched to /status polling.", state: null});
      }
    };

    pollStatus();
    setInterval(pollStatus, 5000);
  </script>
</body>
</html>
""")

    async def handle_ws(websocket: WebSocket):
        await websocket.accept()
        send_lock = asyncio.Lock()
        state["websocket_clients_total"] += 1
        state["websocket_clients_active"] += 1
        await emit("client", f"connected active={state['websocket_clients_active']}")
        try:
            while True:
                message = await websocket.receive_text()
                query = json.loads(message)
                if query.get("action") == "terminate" and query.get("terminateId"):
                    tid = query["terminateId"]
                    finish_pending_query(tid)
                    await write_to_katago({"id": f"terminate-{tid}-{time.time_ns()}", "action": "terminate", "terminateId": tid})
                    continue
                query_id = query.get("id") or str(time.time_ns())
                query["id"] = query_id
                await terminate_pending_query(query_id, "replaced by newer query with the same id")
                pending[query_id] = (websocket, send_lock)
                pending_expected_turns[query_id] = {int(turn) for turn in (query.get("analyzeTurns") or [])}
                pending_completed_turns[query_id] = set()
                state["queries_received"] += 1
                state["last_query_id"] = query_id
                await emit("query", f"Received query id={query_id}, moves={len(query.get('moves', []))}, maxVisits={query.get('maxVisits')}")
                await write_to_katago(query)
        except WebSocketDisconnect:
            await emit("client", "disconnected")
        except Exception as exc:
            await emit("error", f"WebSocket error: {exc!r}")
        finally:
            state["websocket_clients_active"] = max(0, state["websocket_clients_active"] - 1)
            dead = [qid for qid, item in pending.items() if item[0] is websocket]
            for qid in dead:
                finish_pending_query(qid)
                try:
                    await write_to_katago({"id": f"terminate-{qid}-{time.time_ns()}", "action": "terminate", "terminateId": qid})
                except Exception:
                    pass
            await emit("client", f"cleanup active={state['websocket_clients_active']}")

    @web_app.websocket("/")
    async def websocket_root(websocket: WebSocket):
        await handle_ws(websocket)

    @web_app.websocket(KATAGO_WS_PATH)
    async def websocket_katago(websocket: WebSocket):
        await handle_ws(websocket)

    return web_app


## 4. Deploy

This builds and deploys the Modal app, triggers `/monitor` so Modal starts immediately, streams live startup logs until KataGo is ready, then prints the final monitor and WebSocket URLs at the end like the Colab notebook.

The first time someone opens the Modal URL, KataGo/model files download into the Modal Volume named `katago`. Later starts should reuse the cached files.


In [ ]:
import os, queue, re, subprocess, threading, time, urllib.request
from pathlib import Path

APP_NAME_OVERRIDE = ""  # optional; leave blank to read the app name from modal_katago_server.py
STARTUP_LOG_TIMEOUT_SECONDS = 900
KEEP_STREAMING_AFTER_READY = False  # set True if you want logs to keep running after URLs print
TRIGGER_APP_AFTER_DEPLOY = True

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"

def stream_command(cmd):
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    return proc.wait(), lines

def trigger_app(base_url):
    # Modal starts the container on first web request, not merely on deploy.
    url = base_url.rstrip("/") + "/monitor"
    print(f"\nTriggering Modal cold start by opening: {url}")
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                print(f"Monitor responded with HTTP {response.status}")
                return
        except Exception as exc:
            print(f"Monitor trigger attempt {attempt} still starting: {exc!r}")
            time.sleep(5)

def detect_app_name(script_text):
    app_name_match = re.search(r"^APP_NAME\s*=\s*['\"]([^'\"]+)['\"]", script_text, re.MULTILINE)
    if app_name_match:
        return app_name_match.group(1)
    modal_app_match = re.search(r"modal\.App\(['\"]([^'\"]+)['\"]\)", script_text)
    if modal_app_match:
        return modal_app_match.group(1)
    return "katago"

def start_log_reader(app_name):
    proc = subprocess.Popen(
        ["modal", "app", "logs", app_name],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    q = queue.Queue()

    def reader():
        for line in proc.stdout:
            q.put(line)
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()
    return proc, q

script_path = Path.cwd() / "modal_katago_server.py"
if not script_path.exists():
    raise FileNotFoundError(
        f"{script_path} does not exist. Run the 'Write the Modal App' cell first; "
        "it should start with %%writefile modal_katago_server.py."
    )

script_text = script_path.read_text()
APP_NAME = APP_NAME_OVERRIDE.strip() or detect_app_name(script_text)

print("Deploying Modal app...")
print("Using app file:", script_path)
print("Using Modal app name for logs:", APP_NAME)
return_code, deploy_lines = stream_command(["modal", "deploy", str(script_path)])
if return_code != 0:
    raise RuntimeError(f"modal deploy failed with exit code {return_code}")

modal_urls = []
for line in deploy_lines:
    for url in re.findall(r"https://[^\s]+?\.modal\.run", line):
        clean = url.rstrip(".,)];'")
        if clean not in modal_urls:
            modal_urls.append(clean)

base = modal_urls[-1].rstrip("/") if modal_urls else ""
if not base:
    print("\nDeploy finished, but I could not auto-detect the Modal URL.")
    print("Look above for a https://...modal.run URL, then add /monitor or convert it to wss://.../katago.")

print("\n\nLIVE MODAL STARTUP LOGS")
print("Waiting for KataGo startup. This is the Modal equivalent of the Colab live log tail.")
log_proc, log_queue = start_log_reader(APP_NAME)

if TRIGGER_APP_AFTER_DEPLOY and base:
    threading.Thread(target=trigger_app, args=(base,), daemon=True).start()

ready = False
failed = False
start_time = time.time()
try:
    while True:
        if time.time() - start_time > STARTUP_LOG_TIMEOUT_SECONDS:
            print(f"\nStartup log wait timed out after {STARTUP_LOG_TIMEOUT_SECONDS}s.")
            break
        try:
            line = log_queue.get(timeout=1)
        except queue.Empty:
            continue
        if line is None:
            print("\nModal log stream ended.")
            break
        print(line, end="")
        lower = line.lower()
        if "katago ready" in lower or "started, ready to begin handling requests" in lower:
            ready = True
            if not KEEP_STREAMING_AFTER_READY:
                break
        if "katago startup failed" in lower:
            failed = True
            if not KEEP_STREAMING_AFTER_READY:
                break
except KeyboardInterrupt:
    print("\nStopped live log streaming.")
finally:
    if not KEEP_STREAMING_AFTER_READY:
        log_proc.terminate()

print("\n\nSETUP COMPLETE" if ready else "\n\nSETUP STOPPED BEFORE READY")
if failed:
    print("A startup error appeared in the logs above. Fix that, then rerun Write the Modal App and Deploy.")

if base:
    print("\nFinal KaTrain/SWHub WebSocket URL:")
    print(base.replace("https://", "wss://").replace("http://", "ws://") + "/katago")
    print("\nLive Monitor URL:")
    print(base + "/monitor")
else:
    print("\nNo Modal URL was auto-detected from deploy output.")


## 5. Convert The URL

The deploy cell should print both URLs automatically. If you still need to convert a URL manually, Modal URLs look like:

```text
https://YOUR-WORKSPACE--katago-fastapi-app.modal.run
```

Paste that URL below and run the cell. It will print the exact KaTrain/SWHub URL.


In [ ]:
MODAL_HTTPS_URL = ""  # paste your https://...modal.run URL here

if not MODAL_HTTPS_URL.strip():
    print("Paste your Modal https://...modal.run URL into MODAL_HTTPS_URL, then rerun this cell.")
else:
    base = MODAL_HTTPS_URL.strip().rstrip("/")
    print("Open monitor in browser:")
    print(base + "/monitor")
    print()
    print("Paste this into KaTrain/SWHub remote engine settings:")
    print(base.replace("https://", "wss://").replace("http://", "ws://") + "/katago")


## 6. Live Logs / Check Status

After you have a Modal URL, you can also open:

```text
https://YOUR-URL.modal.run/status
https://YOUR-URL.modal.run/logs
https://YOUR-URL.modal.run/monitor
```

The Deploy cell now streams live logs automatically. If you need logs later, run `modal app logs KataGo-Analyse` in a notebook cell. If KaTrain/SWHub fails to connect, open `/monitor` first. If the monitor is still tuning or downloading, wait.


## 7. Stop When Done

This prevents accidental ongoing warm-container usage. The cached KataGo files stay in the Modal Volume for next time.


In [ ]:
# Uncomment when you want to stop the deployed app.
# !modal app stop KataGo-Analyse


## Troubleshooting

- **Want cheaper/slower testing?** Set `ENGINE_MODE = "cpu"`, rerun **Write the Modal App**, then rerun **Deploy**.
- **Want the older convolutional baseline?** Set `MAIN_MODEL_PRESET = "modal_fast_b18"`, rerun **Write the Modal App**, then rerun **Deploy**.
- **GPU not available?** Try `MODAL_GPU = "L4"` or `MODAL_GPU = "A10"`.
- **KaTrain connects but no result?** Open `/monitor`; KataGo may still be downloading, loading, or tuning.
- **Spending credits too fast?** Use `MODAL_GPU = "T4"`, keep the compact transformer, disable Human SL if rank-style features are unnecessary, and stop the app when done.
